# 05.1 — Retrieval and grounding pipelines

Unit [02.2](../../02_genai_and_agents/02_rag_grounding/README.md) built RAG by hand:
you chunked in Python, embedded in Python, and pushed documents. This lab hands all
of that to Azure AI Search and builds the **indexer pipeline**:

```
data source  →  indexer  →  skillset  →  index (+ vectorizer)
  where           the job     what happens    the schema
```

⚠️ **Cost.** Azure AI Search bills **by the hour** from the moment the service
exists — a Basic service is ~$75/month whether you query it or not. The final cell
deletes every index, indexer, skillset, and data source this lab creates.
**That does not stop the hourly service charge.** Only deleting the service does.

Everything else here costs cents.

**Prerequisites:** `03_provision_labs.ps1` has run (Search + Storage exist), the
search service has a system-assigned identity with *Storage Blob Data Reader* on
the storage account and *Cognitive Services OpenAI User* on the Foundry resource,
and you hold *Search Service Contributor* + *Search Index Data Contributor*.

## 1. Configuration

All artifacts go to a git-ignored `lab_output/` folder next to this notebook.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import (
    cfg,
    credential,
    chat_client,
    project_client,
    search_index_client,
    search_client,
    blob_service_client,
    embed,
)

OUT = pathlib.Path.cwd() / "lab_output"
OUT.mkdir(exist_ok=True)
(OUT / ".gitignore").write_text("*\n", encoding="utf-8")

# Every object this lab creates carries this prefix so cleanup is unambiguous.
PREFIX = "ai103-51"
INDEX_NAME = f"{PREFIX}-chunks"
DATASOURCE_NAME = f"{PREFIX}-ds"
SKILLSET_NAME = f"{PREFIX}-ss"
INDEXER_NAME = f"{PREFIX}-ixr"
OCR_SKILLSET_NAME = f"{PREFIX}-ocr-ss"
CONTAINER = f"{PREFIX}-src"

EMBED_DEPLOYMENT = cfg.require("MODEL_EMBEDDING")
EMBED_MODEL = "text-embedding-3-small"
DIMS = 1536  # text-embedding-3-small. -large is 3072. This number must match everywhere.

print("search  :", cfg.require("AZURE_SEARCH_ENDPOINT", unit="05.1"))
print("storage :", cfg.require("AZURE_STORAGE_ACCOUNT"))
print("foundry :", cfg.require("AZURE_OPENAI_ENDPOINT"))
print("output  :", OUT)

## 2. Generate the corpus

The lab makes its own source documents so nothing depends on files that may not
exist. Three kinds, deliberately:

| File | Why |
|---|---|
| `.md` policy documents | Native text cracking, heading structure |
| A hand-built one-page `.pdf` | Exercises the blob indexer's PDF cracker |
| A `.png` with text drawn on it | There is **no** extractable text layer — only OCR can read it |

The PNG matters: it is the difference between "the indexer read the file" and "the
indexer read the *content*".

In [ ]:
DOCS = {
    "returns-policy.md": """# Contoso Returns Policy

## Standard returns
Unopened items may be returned within 30 days of delivery for a full refund.
The original packaging must be intact and the receipt must be presented.

## Opened items
Opened items may be returned within 14 days of delivery. A restocking fee of
15 percent applies. Items showing physical damage are not eligible.

## Disputes
A dispute about a refused return must be filed within 14 days of the refusal.
Disputes are handled by the Customer Resolution team and resolved within 10
business days.
""",
    "warranty-cx4400.md": """# CX-4400 Industrial Controller — Warranty

## Coverage
The CX-4400 carries a 36 month limited warranty from the date of commissioning.
Coverage includes the mainboard, the power supply module, and firmware defects.

## Exclusions
Warranty is void if the enclosure seal is broken, if the unit is operated above
60 degrees Celsius ambient, or if non-Contoso firmware is flashed.

## Replacement units
Advance replacement is available for units under contract SLA-GOLD. The failed
unit must be returned within 21 days or the replacement is invoiced at list price.
""",
    "shipping-faq.md": """# Shipping FAQ

## How long does delivery take?
Standard delivery is 3 to 5 business days. Express delivery is next business day
if ordered before 14:00 local time.

## Can I change the delivery address?
The address can be changed until the parcel is scanned at the outbound depot.
After that the parcel must be refused on delivery and returned to sender.

## Do you ship internationally?
Yes, to 41 countries. Duties and import taxes are payable by the recipient.
""",
}

for name, body in DOCS.items():
    (OUT / name).write_text(body, encoding="utf-8")

print(f"wrote {len(DOCS)} markdown documents")

In [ ]:
def build_pdf(path: pathlib.Path, lines: list[str]) -> None:
    """Write a minimal, valid one-page PDF with a real text layer.

    Hand-rolled so the lab has no extra dependency. The point is only that the
    blob indexer can crack it and recover the text without OCR.
    """
    stream_parts = ["BT", "/F1 12 Tf", "56 740 Td", "16 TL"]
    for line in lines:
        safe = line.replace("\\", r"\\").replace("(", r"\(").replace(")", r"\)")
        stream_parts.append(f"({safe}) Tj T*")
    stream_parts.append("ET")
    stream = "\n".join(stream_parts).encode("latin-1")

    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
        b"/Resources << /Font << /F1 5 0 R >> >> /Contents 4 0 R >>",
        b"<< /Length " + str(len(stream)).encode() + b" >>\nstream\n" + stream + b"\nendstream",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
    ]

    buf = bytearray(b"%PDF-1.4\n")
    offsets = []
    for i, obj in enumerate(objects, start=1):
        offsets.append(len(buf))
        buf += f"{i} 0 obj\n".encode() + obj + b"\nendobj\n"

    xref_at = len(buf)
    buf += f"xref\n0 {len(objects) + 1}\n".encode()
    buf += b"0000000000 65535 f \n"
    for off in offsets:
        buf += f"{off:010d} 00000 n \n".encode()
    buf += (
        f"trailer\n<< /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref_at}\n".encode()
        + b"%%EOF\n"
    )
    path.write_bytes(bytes(buf))


build_pdf(
    OUT / "sla-gold.pdf",
    [
        "Contoso Support Contract SLA-GOLD",
        "",
        "Response time: 2 hours, 24x7, for severity 1 incidents.",
        "Advance hardware replacement is included at no charge.",
        "Onsite engineer dispatch is available within 8 hours in",
        "metropolitan service areas.",
        "",
        "Escalation contact: the named Technical Account Manager.",
        "Credits for missed response targets are 5 percent of the",
        "monthly contract value per missed incident, capped at 25 percent.",
    ],
)
print("wrote sla-gold.pdf:", (OUT / "sla-gold.pdf").stat().st_size, "bytes")

In [ ]:
from PIL import Image, ImageDraw

# A scanned-notice lookalike. Pixels only: no text layer, no metadata, nothing an
# ordinary document cracker can recover. This is what the OCR skill exists for.
img = Image.new("RGB", (900, 420), "white")
d = ImageDraw.Draw(img)
d.rectangle([12, 12, 888, 408], outline="black", width=3)
for i, line in enumerate(
    [
        "CONTOSO SERVICE NOTICE",
        "",
        "Depot maintenance window: 02:00 to 05:00 UTC on Sunday.",
        "Order reference CX-4400-RMA is on hold pending inspection.",
        "Contact the Customer Resolution team for expedited handling.",
    ]
):
    d.text((48, 60 + i * 56), line, fill="black")
img.save(OUT / "service-notice.png")

print("wrote service-notice.png")
print("NOTE: the default PIL font is small. OCR will still read it, but if you want")
print("      a realistic scan, load a TrueType font with ImageFont.truetype().")
img

## 3. Upload to blob storage

The indexer reads from a container, not from your laptop. Uploads use
`DefaultAzureCredential` — you need **Storage Blob Data Contributor** on the
account.

In [ ]:
bsc = blob_service_client()
try:
    bsc.create_container(CONTAINER)
    print("container created:", CONTAINER)
except Exception as exc:  # already exists is fine
    print("container:", CONTAINER, "-", type(exc).__name__)

container = bsc.get_container_client(CONTAINER)
uploaded = []
for f in sorted(OUT.iterdir()):
    if f.suffix.lower() not in {".md", ".pdf", ".png"}:
        continue
    with f.open("rb") as fh:
        container.upload_blob(name=f.name, data=fh, overwrite=True)
    uploaded.append(f.name)

print("uploaded:", ", ".join(uploaded))

## 4. Object 1 — the data source

The data source owns **where** and **how to authenticate**, and nothing else.

Keyless connection strings for blob data sources use the
`ResourceId=/subscriptions/.../storageAccounts/<name>;` form: no key, no SAS. Search
then authenticates with its **own managed identity**, which is why the search
service needs *Storage Blob Data Reader* on the account. This is the pattern the
exam wants when it says "keyless credentials".

The **deletion detection policy** is the other thing this object owns: without one,
deleting a source blob leaves its chunks in the index forever.

In [ ]:
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataSourceType,
    NativeBlobSoftDeleteDeletionDetectionPolicy,
)

indexer_client = SearchIndexerClient(
    endpoint=cfg["AZURE_SEARCH_ENDPOINT"], credential=credential()
)

resource_id = (
    f"/subscriptions/{cfg['AZURE_SUBSCRIPTION_ID']}"
    f"/resourceGroups/{cfg['AZURE_RESOURCE_GROUP']}"
    f"/providers/Microsoft.Storage/storageAccounts/{cfg['AZURE_STORAGE_ACCOUNT']}"
)

data_source = SearchIndexerDataSourceConnection(
    name=DATASOURCE_NAME,
    type=SearchIndexerDataSourceType.AZURE_BLOB,
    connection_string=f"ResourceId={resource_id};",  # managed identity, no key
    container=SearchIndexerDataContainer(name=CONTAINER),
    # Requires blob soft delete on the storage account; harmless if not enabled.
    data_deletion_detection_policy=NativeBlobSoftDeleteDeletionDetectionPolicy(),
)

indexer_client.create_or_update_data_source_connection(data_source)
print("data source created:", DATASOURCE_NAME)

## 5. Object 2 — the index (schema + vectorizer)

This is a **chunk-level** index: one document per chunk, not per file. `parent_id`
carries the file identity so citations still point at a real document.

Read the attribute choices below deliberately — each is a decision the exam tests:

| Field | Attributes | Why |
|---|---|---|
| `chunk_id` | `key`, `filterable`, analyzer `keyword` | Keys must be one token; `keyword` stops the tokenizer shredding it |
| `parent_id` | `filterable` only | You filter and group by it; you never full-text search it |
| `title` | `searchable`, `filterable` | Matched in queries *and* usable in `$filter` |
| `chunk` | `searchable` only | The body text. Not filterable — filtering free text is meaningless and costs index size |
| `chunk_vector` | `searchable` + vector profile, `stored=False` | A vector field must be searchable to be queried. `stored=False` drops the raw floats from the response payload and roughly halves storage — and it is **immutable** |
| `doc_type` | `filterable`, `facetable` | Low cardinality, good facet |

The **vectorizer** at the bottom is the half of integrated vectorization that runs
at *query* time. Without it you cannot send `"kind": "text"` vector queries.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmMetric,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SearchField(
            name="chunk_id",
            type=SearchFieldDataType.String,
            key=True,
            filterable=True,
            sortable=True,
            analyzer_name="keyword",
        ),
        SearchField(name="parent_id", type=SearchFieldDataType.String, filterable=True),
        SearchField(
            name="title", type=SearchFieldDataType.String, searchable=True, filterable=True
        ),
        SearchField(name="chunk", type=SearchFieldDataType.String, searchable=True),
        SearchField(
            name="doc_type", type=SearchFieldDataType.String, filterable=True, facetable=True
        ),
        SearchField(
            name="chunk_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            stored=False,  # immutable: you cannot flip this later
            vector_search_dimensions=DIMS,
            vector_search_profile_name="hnsw-aoai",
        ),
    ],
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw",
                parameters=HnswParameters(
                    m=4, ef_construction=400, ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE,
                ),
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="hnsw-aoai",
                algorithm_configuration_name="hnsw",
                vectorizer_name="aoai-vectorizer",
            )
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="aoai-vectorizer",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=cfg["AZURE_OPENAI_ENDPOINT"],
                    deployment_name=EMBED_DEPLOYMENT,
                    model_name=EMBED_MODEL,
                    # No api_key: the search service's managed identity calls the model.
                ),
            )
        ],
    ),
    semantic_search=SemanticSearch(
        default_configuration_name="default",
        configurations=[
            SemanticConfiguration(
                name="default",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="chunk")],
                ),
            )
        ],
    ),
)

search_index_client().create_or_update_index(index)
print("index created:", INDEX_NAME, f"({DIMS}-dim vectors)")

## 6. Object 3 — the skillset

Two skills and one projection.

- **`SplitSkill`** chunks `/document/content` into `/document/pages/*`.
  **This is where chunk size and overlap live.** Not the index, not the indexer.
- **`AzureOpenAIEmbeddingSkill`** runs with `context="/document/pages/*"`, so it
  executes once per chunk and writes a vector per chunk.
- **Index projections** turn each `/document/pages/*` into its own index document.
  `projection_mode="skipIndexingParentDocuments"` says: index the chunks, throw the
  parent away.

Watch the `context` / `source` paths. If they disagree the pipeline succeeds and
produces nothing — Search does not error on a path that does not exist.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexerSkillset,
    SplitSkill,
    AzureOpenAIEmbeddingSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
)

split = SplitSkill(
    name="split",
    description="Chunk the cracked document text",
    context="/document",
    text_split_mode="pages",       # 'pages' = fixed-size blocks; 'sentences' also exists
    maximum_page_length=1200,      # characters
    page_overlap_length=200,       # ~17% overlap
    unit="characters",
    default_language_code="en",
    inputs=[InputFieldMappingEntry(name="text", source="/document/content")],
    outputs=[OutputFieldMappingEntry(name="textItems", target_name="pages")],
)

embedding = AzureOpenAIEmbeddingSkill(
    name="embed",
    description="Vectorise each chunk",
    context="/document/pages/*",   # once per chunk
    resource_url=cfg["AZURE_OPENAI_ENDPOINT"],
    deployment_name=EMBED_DEPLOYMENT,
    model_name=EMBED_MODEL,
    dimensions=DIMS,
    inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
    outputs=[OutputFieldMappingEntry(name="embedding", target_name="chunk_vector")],
)

projection = SearchIndexerIndexProjection(
    selectors=[
        SearchIndexerIndexProjectionSelector(
            target_index_name=INDEX_NAME,
            parent_key_field_name="parent_id",
            source_context="/document/pages/*",
            mappings=[
                InputFieldMappingEntry(name="chunk", source="/document/pages/*"),
                InputFieldMappingEntry(
                    name="chunk_vector", source="/document/pages/*/chunk_vector"
                ),
                InputFieldMappingEntry(name="title", source="/document/metadata_storage_name"),
                InputFieldMappingEntry(
                    name="doc_type", source="/document/metadata_storage_file_extension"
                ),
            ],
        )
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS
    ),
)

skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    description="Split + embed, projected to a chunk index",
    skills=[split, embedding],
    index_projection=projection,
)

indexer_client.create_or_update_skillset(skillset)
print("skillset created:", SKILLSET_NAME)
print("chunk size:", split.maximum_page_length, "chars, overlap:", split.page_overlap_length)

## 7. Object 4 — the indexer

The job. It owns the schedule, batch size, error tolerance, field mappings, and the
parsing configuration.

Two configuration values matter more than the rest:

- **`dataToExtract: "contentAndMetadata"`** — pull the text *and* the blob metadata
  (`metadata_storage_name`, `_path`, `_file_extension`, …). The projection above
  depends on those.
- **`imageAction: "generateNormalizedImages"`** — extract embedded and standalone
  images into `/document/normalized_images/*`. **Without this the OCR skill has
  nothing to run on**, and image files index as empty documents. It also requires an
  attached AI Services resource once you actually add a vision skill.

`max_failed_items=-1` means "do not abort the run because one blob is unreadable" —
the right setting for a heterogeneous corpus, and the wrong one if silent data loss
is unacceptable.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexer,
    IndexingParameters,
    IndexingParametersConfiguration,
    IndexingSchedule,
)
import datetime as dt

indexer = SearchIndexer(
    name=INDEXER_NAME,
    data_source_name=DATASOURCE_NAME,
    target_index_name=INDEX_NAME,
    skillset_name=SKILLSET_NAME,
    parameters=IndexingParameters(
        batch_size=10,
        max_failed_items=-1,
        max_failed_items_per_batch=-1,
        configuration=IndexingParametersConfiguration(
            parsing_mode="default",
            data_to_extract="contentAndMetadata",
            image_action="generateNormalizedImages",
            query_timeout=None,
        ),
    ),
    # Schedules live here, on the indexer. Minimum interval is 5 minutes.
    schedule=IndexingSchedule(interval=dt.timedelta(hours=24)),
)

indexer_client.create_or_update_indexer(indexer)
print("indexer created:", INDEXER_NAME)

In [ ]:
import time

indexer_client.run_indexer(INDEXER_NAME)
print("run requested; polling execution status\n")

for attempt in range(40):
    status = indexer_client.get_indexer_status(INDEXER_NAME)
    last = status.last_result
    state = last.status if last else "(no run yet)"
    print(f"  [{attempt:02d}] {state}")
    if last and last.status in ("success", "transientFailure", "persistentFailure"):
        print("\nitems processed:", last.item_count, " failed:", last.failed_item_count)
        if last.error_message:
            print("error:", last.error_message)
        for err in (last.errors or [])[:5]:
            print("  doc error:", err.key, "->", err.error_message[:200])
        for warn in (last.warnings or [])[:5]:
            print("  warning  :", warn.key, "->", warn.message[:200])
        break
    time.sleep(6)
else:
    print("still running - re-run this cell")

> **This is the ingestion-monitoring surface.** `get_indexer_status()` returns the
> same data the portal's **Execution history** blade shows: per-run item counts,
> per-document errors keyed by document, and warnings. Unit
> [01.3](../../01_plan_and_manage/03_manage_monitor_secure/README.md) asks you to
> "monitor data ingestion quality and index health" — this call, plus
> `get_document_count()`, is the answer.

If the run failed with a 403 to Storage or to the embedding deployment, the
**search service's** identity is missing a role — not yours. See the README's
troubleshooting section. The next cell gives you a working index either way so the
rest of the lab still runs.

In [ ]:
sc = search_client(INDEX_NAME)
time.sleep(3)
count = sc.get_document_count()
print("documents (chunks) in index:", count)

if count == 0:
    print("\nIndexer produced nothing. Falling back to a client-side push so the")
    print("retrieval section still works. This is exactly the unit 02.2 pattern.\n")

    import re, uuid

    fallback = []
    for name, body in DOCS.items():
        for part in [p.strip() for p in re.split(r"\n(?=## )", body) if p.strip()]:
            fallback.append(
                {
                    "chunk_id": uuid.uuid4().hex,
                    "parent_id": name,
                    "title": name,
                    "doc_type": ".md",
                    "chunk": f"{name} > {part}",
                }
            )
    for chunk, vec in zip(fallback, embed([c["chunk"] for c in fallback])):
        chunk["chunk_vector"] = vec

    result = sc.upload_documents(documents=fallback)
    print(f"pushed {sum(1 for r in result if r.succeeded)}/{len(fallback)} chunks")
    time.sleep(3)
    print("documents in index:", sc.get_document_count())

## 8. Enrichment for images: the OCR skillset

`service-notice.png` is pixels. The indexer cracked it and found no text, so it
contributed nothing above. To read it you need a vision skill, and vision skills
are **billable AI enrichments**, which is why the skillset must carry a
`cognitive_services_account`.

Two ways to attach it:

| Attachment | Class | Requires |
|---|---|---|
| Managed identity (preferred) | `AIServicesAccountIdentity` | Search service identity holds **Cognitive Services User** on the Foundry resource |
| Key | `CognitiveServicesAccountKey` | A key you must rotate |

The pipeline shape for image-bearing documents is always the same three moves:

1. **OCR skill** over `/document/normalized_images/*` → text per image
2. **Image Analysis skill** over the same context → tags/captions, so the *visual*
   content is searchable too, not just text printed on it
3. **Merge skill** to splice the OCR text back into `/document/content` at the right
   offsets, producing one coherent text field to chunk

The cell below builds all three and attempts to create the skillset. If your
environment lacks the role assignment it prints the failure rather than pretending;
the definitions are the exam material either way.

In [ ]:
from azure.search.documents.indexes.models import (
    OcrSkill,
    ImageAnalysisSkill,
    MergeSkill,
    VisualFeature,
    AIServicesAccountIdentity,
)

ocr = OcrSkill(
    name="ocr",
    description="Read text printed inside images",
    context="/document/normalized_images/*",
    default_language_code="en",
    should_detect_orientation=True,
    inputs=[InputFieldMappingEntry(name="image", source="/document/normalized_images/*")],
    outputs=[OutputFieldMappingEntry(name="text", target_name="ocr_text")],
)

image_analysis = ImageAnalysisSkill(
    name="image-analysis",
    description="Describe what the image shows, not just what it says",
    context="/document/normalized_images/*",
    default_language_code="en",
    visual_features=[VisualFeature.TAGS, VisualFeature.DESCRIPTION],
    inputs=[InputFieldMappingEntry(name="image", source="/document/normalized_images/*")],
    outputs=[
        OutputFieldMappingEntry(name="tags", target_name="image_tags"),
        OutputFieldMappingEntry(name="description", target_name="image_description"),
    ],
)

merge = MergeSkill(
    name="merge",
    description="Splice OCR text back into the document text at the correct offsets",
    context="/document",
    insert_pre_tag=" ",
    insert_post_tag=" ",
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/content"),
        InputFieldMappingEntry(
            name="itemsToInsert", source="/document/normalized_images/*/ocr_text"
        ),
        InputFieldMappingEntry(
            name="offsets", source="/document/normalized_images/*/contentOffset"
        ),
    ],
    outputs=[OutputFieldMappingEntry(name="mergedText", target_name="merged_content")],
)

ocr_skillset = SearchIndexerSkillset(
    name=OCR_SKILLSET_NAME,
    description="Multimodal: OCR + image analysis + merge, then split and embed",
    skills=[ocr, image_analysis, merge, split, embedding],
    # Vision skills are billable enrichments: they REQUIRE this attachment.
    cognitive_services_account=AIServicesAccountIdentity(
        subdomain_url=cfg["AZURE_OPENAI_ENDPOINT"],
        identity=None,  # None = the search service's system-assigned identity
    ),
    index_projection=projection,
)

try:
    indexer_client.create_or_update_skillset(ocr_skillset)
    print("OCR skillset created:", OCR_SKILLSET_NAME)
    print("To use it: point the indexer's skillset_name at it, set the split skill's")
    print("input source to /document/merged_content, and re-run.")
except Exception as exc:
    print("could not create the OCR skillset:", type(exc).__name__)
    print(str(exc)[:600])
    print("\nMost likely the search service identity lacks 'Cognitive Services User'")
    print("on the Foundry resource. The definitions above are still the exam answer.")

> **Exam note.** After adding the merge skill you must also **repoint the split
> skill** at `/document/merged_content` instead of `/document/content`. Forgetting
> this is the single most common reason an OCR pipeline "runs fine" and still
> cannot answer questions about text in images: the OCR output exists in the
> enriched tree but never reaches the index.

## 9. Where your own logic goes: the custom Web API skill

Anything the built-in catalogue does not do — a proprietary classifier, a lookup
against your CRM, a redaction pass — goes in a `WebApiSkill`. Search POSTs batches
to your HTTPS endpoint and expects an exact response envelope.

We define it and print it rather than deploying a function app; the contract is
what is tested.

In [ ]:
import json
from azure.search.documents.indexes.models import WebApiSkill

custom = WebApiSkill(
    name="classify-sensitivity",
    description="Call our own classifier for each chunk",
    context="/document/pages/*",
    uri="https://contoso-enrich.azurewebsites.net/api/classify",
    http_method="POST",
    timeout="PT30S",           # ISO-8601 duration, max PT230S
    batch_size=10,             # how many records per HTTP call
    degree_of_parallelism=2,
    # Entra ID auth to your own API: the search service's identity requests a token
    # for this app registration. Preferred over http_headers with a shared secret.
    auth_resource_id="api://contoso-enrich",
    inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
    outputs=[OutputFieldMappingEntry(name="sensitivity", target_name="sensitivity")],
)

print("Skill definition:")
print(json.dumps(custom.as_dict(), indent=2))

print("\nSearch sends:")
print(json.dumps({"values": [{"recordId": "0", "data": {"text": "Opened items ..."}}]}, indent=2))
print("\nYour endpoint MUST return exactly this shape (recordId echoed back):")
print(
    json.dumps(
        {
            "values": [
                {
                    "recordId": "0",
                    "data": {"sensitivity": "internal"},
                    "errors": [],
                    "warnings": [],
                }
            ]
        },
        indent=2,
    )
)

## 10. The knowledge store

Index projections send enrichments to **another index**. A **knowledge store** sends
them to **Azure Storage** — tables for Power BI and analytics, blobs for the raw
enriched JSON, files for extracted images.

They are not alternatives to each other; a skillset can have both. The exam
contrast is *destination and purpose*: index projections serve **retrieval**, the
knowledge store serves **downstream analysis of the enrichment itself**.

The cell prints the definition rather than creating it, because a knowledge store
needs a storage connection string and writes real tables you would then have to
clean up.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexerKnowledgeStore,
    SearchIndexerKnowledgeStoreProjection,
    SearchIndexerKnowledgeStoreTableProjectionSelector,
    SearchIndexerKnowledgeStoreObjectProjectionSelector,
)

knowledge_store = SearchIndexerKnowledgeStore(
    storage_connection_string=f"ResourceId={resource_id};",  # keyless
    projections=[
        SearchIndexerKnowledgeStoreProjection(
            # Tables: relational slices, one row per chunk, joinable in Power BI
            tables=[
                SearchIndexerKnowledgeStoreTableProjectionSelector(
                    table_name="ai103Chunks",
                    generated_key_name="chunk_key",
                    source_context="/document/pages/*",
                    inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
                )
            ],
            # Objects: the whole enriched document as JSON blobs
            objects=[
                SearchIndexerKnowledgeStoreObjectProjectionSelector(
                    storage_container="ai103-knowledgestore",
                    generated_key_name="doc_key",
                    source_context="/document",
                    inputs=[InputFieldMappingEntry(name="content", source="/document/content")],
                )
            ],
        )
    ],
)

print(json.dumps(knowledge_store.as_dict(), indent=2)[:1500])
print("\n(attach with SearchIndexerSkillset(..., knowledge_store=knowledge_store))")

## 11. Four retrieval modes on the same queries

Two queries chosen to break in opposite directions:

- **`"CX-4400"`** — a rare literal. BM25 nails it; embeddings have never seen this
  token and will happily return "industrial controller" prose that does not contain
  it.
- **`"how long do I have to send something back after opening it"`** — pure
  paraphrase. The words "return", "refund", and "14 days" never appear in the query.

Note `VectorizableTextQuery`: because the index carries a **vectorizer**, we send
*text* and Search embeds it. No `embed()` call, no dimension mismatch, no second
place to update when the model changes.

In [ ]:
from azure.search.documents.models import VectorizableTextQuery, QueryType, QueryCaptionType

SELECT = ["chunk_id", "parent_id", "title", "chunk"]


def show(label, results):
    print(f"\n{label}")
    print("-" * 78)
    empty = True
    for r in results:
        empty = False
        rr = r.get("@search.reranker_score")
        extra = f"  rerank={rr:.2f}" if rr is not None else ""
        snippet = " ".join(r["chunk"].split())[:78]
        print(f"  {r['@search.score']:>7.4f}{extra}  {r['title']}")
        print(f"           {snippet}")
    if empty:
        print("  (no results)")


def keyword(q, k=3):
    return sc.search(search_text=q, top=k, select=SELECT)


def vector(q, k=3):
    vq = VectorizableTextQuery(text=q, k_nearest_neighbors=k, fields="chunk_vector")
    return sc.search(search_text=None, vector_queries=[vq], top=k, select=SELECT)


def hybrid(q, k=3):
    vq = VectorizableTextQuery(text=q, k_nearest_neighbors=k * 5, fields="chunk_vector")
    return sc.search(search_text=q, vector_queries=[vq], top=k, select=SELECT)


def hybrid_semantic(q, k=3):
    vq = VectorizableTextQuery(text=q, k_nearest_neighbors=50, fields="chunk_vector")
    return sc.search(
        search_text=q,
        vector_queries=[vq],
        top=k,
        select=SELECT,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default",
        query_caption=QueryCaptionType.EXTRACTIVE,
    )


for q in [
    "CX-4400",
    "how long do I have to send something back after opening it",
]:
    print("\n" + "=" * 78)
    print("QUERY:", q)
    print("=" * 78)
    show("keyword (BM25)", keyword(q))
    show("vector (index vectorizer embeds the query)", vector(q))
    show("hybrid (RRF fusion)", hybrid(q))
    try:
        show("hybrid + semantic ranker", hybrid_semantic(q))
    except Exception as exc:
        print("\nsemantic ranker unavailable on this tier/region:", type(exc).__name__)

Read the numbers, not the ordering. `@search.score` under BM25 is unbounded and
term-frequency driven; under pure vector it is a cosine similarity in 0–1; under
hybrid it is an **RRF score** — typically around $1/61 \approx 0.016$ per
contributing ranker, so a document that placed well in both lists lands near 0.03.
None of those three are comparable to each other, and none are comparable to
`@search.rerankerScore`, which is 0–4.

> **Exam trap.** "Set a relevance threshold of 0.7 on `@search.score`" is a
> distractor unless the query is pure vector. Thresholding is only meaningful on
> the reranker score, or on cosine similarity from a pure vector query.

### Semantic captions: the citation snippet

Captions are the reason to use the semantic ranker even when the ordering is
already right — they hand you the exact passage that matched, which is what you
show a user as evidence.

In [ ]:
try:
    for r in hybrid_semantic("what happens if I miss the response target", k=2):
        print(f"{r['title']}  rerank={r.get('@search.reranker_score')}")
        for cap in r.get("@search.captions") or []:
            print("  caption:", " ".join((cap.text or "").split())[:200])
        print()
except Exception as exc:
    print("semantic ranker unavailable:", type(exc).__name__)

## 12. Filters — the free relevance win

Filters are applied **before** ranking, are exact, and cost nothing. They are how
you enforce tenancy, recency, and language, and the first thing to reach for when
retrieval returns plausible-but-wrong documents from the wrong scope.

`doc_type` is `filterable` and `facetable`; `chunk` is neither, by design.

In [ ]:
results = sc.search(
    search_text="replacement",
    filter="doc_type eq '.md'",
    facets=["doc_type"],
    top=3,
    select=SELECT,
)
rows = list(results)
for r in rows:
    print(f"{r['@search.score']:>7.3f}  {r['title']}")
print("\nfacets:", results.get_facets())

# This one FAILS, on purpose: 'chunk' is searchable but not filterable.
try:
    list(sc.search(search_text="*", filter="chunk eq 'anything'", top=1))
except Exception as exc:
    print("\nexpected failure ->", str(exc)[:220])

## 13. Connect the pipeline to an agent

The `AzureAISearchTool` hands the index to a Foundry agent. Three things to notice:

1. It takes a **connection id**, not an endpoint and key. The project's Search
   connection holds the credential; rotating it never touches agent code.
2. The **model decides** whether and when to search, and with what text. You do not
   write a retrieval call.
3. Citations come back as annotations on the message, not as a separate array you
   have to correlate.

If no Search connection exists in your project, create one:
**ai.azure.com → Management center → Connected resources → + New connection →
Azure AI Search**, authentication **Microsoft Entra ID**.

In [ ]:
agent_id = None
thread_id = None
project = project_client()

search_conn = None
for c in project.connections.list():
    if "search" in str(getattr(c, "type", "")).lower():
        search_conn = c
        break

if search_conn is None:
    print("No Azure AI Search connection in this project - create one in the portal")
    print("(Management center -> Connected resources -> + New connection).")
else:
    print("using connection:", search_conn.name, "->", search_conn.id)

    from azure.ai.agents.models import AzureAISearchTool, AzureAISearchQueryType

    tool = AzureAISearchTool(
        index_connection_id=search_conn.id,
        index_name=INDEX_NAME,
        query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
        top_k=5,
    )

    agent = project.agents.create_agent(
        model=cfg["MODEL_MINI"],
        name=f"{PREFIX}-grounded-agent",
        instructions=(
            "You answer only from the Contoso documents returned by your search tool. "
            "Cite the document title for every claim. If the documents do not contain "
            "the answer, reply exactly: NOT_IN_DOCUMENTS."
        ),
        tools=tool.definitions,
        tool_resources=tool.resources,
    )
    agent_id = agent.id
    print("agent created:", agent_id)

In [ ]:
if agent_id:
    thread = project.agents.threads.create()
    thread_id = thread.id

    for question in [
        "If I opened the box, how long do I have to return it and what does it cost me?",
        "Who is the current CEO of Contoso?",  # deliberately not in the corpus
    ]:
        project.agents.messages.create(thread_id=thread_id, role="user", content=question)
        run = project.agents.runs.create_and_process(thread_id=thread_id, agent_id=agent_id)
        print("\nQ:", question)
        print("run status:", run.status)
        if run.status == "failed":
            print("error:", run.last_error)
            continue
        messages = list(project.agents.messages.list(thread_id=thread_id))
        latest = next((m for m in messages if m.role == "assistant"), None)
        if latest and latest.text_messages:
            print("A:", latest.text_messages[0].text.value)
        for ann in getattr(latest, "url_citation_annotations", []) or []:
            print("   citation:", ann.url_citation.title)
else:
    print("skipped - no agent")

The second question should return `NOT_IN_DOCUMENTS`. That behaviour is *prompt
discipline*, not a feature — the tool will happily return weakly-relevant chunks
and the model will happily use them unless you forbid it.

## 14. Agentic retrieval

The `AzureAISearchTool` still does **one search per tool call over the text the
model chose**. Agentic retrieval moves the planning into Search itself: you send
the whole conversation, and the service decomposes it into several subqueries, runs
them in parallel, merges and re-ranks the results, and optionally synthesises a
cited answer.

That is the only thing that works for a question whose meaning depends on a
previous turn — *"and what about the opened ones?"* — because there is no single
similarity search over that literal string that can succeed.

The naming has moved: the first preview called these **knowledge agents** with a
`retrieve` operation; the current API models them as **knowledge bases** built over
**knowledge sources**, with `output_mode` of `extractiveData` (grounding chunks) or
`answerSynthesis` (a written, referenced answer). Expect either term in exam
wording. Both are **preview** and the surface depends on your SDK version, so the
cell below probes rather than assumes.

In [ ]:
sic = search_index_client()
available = sorted(
    m for m in dir(sic) if "knowledge" in m.lower() or "agent" in m.lower()
)

print("agentic-retrieval methods on SearchIndexClient in this SDK version:")
for m in available:
    print("  ", m)
if not available:
    print("   (none - your azure-search-documents predates agentic retrieval)")

print(
    """
Conceptually, whichever name your SDK uses:

  1. Define the knowledge source(s)  -> which index / blob / SQL to draw on,
                                        plus the embedding model for ingestion
  2. Define the knowledge base/agent -> which chat model plans the subqueries,
                                        output_mode = extractiveData | answerSynthesis,
                                        reranker threshold, max output size
  3. Send the whole message list     -> the service plans, searches N times in
                                        parallel, merges, re-ranks, and returns
                                        grounding data + per-subquery activity

You are billed for the planning tokens as well as the searches. That is the
trade: more cost and latency per turn, in exchange for multi-hop questions that
single-shot retrieval simply cannot answer.
"""
)

---

## Exercise

Solutions are in [quiz.md](quiz.md).

1. **Make the image searchable.** Repoint the indexer at `OCR_SKILLSET_NAME`, change
   the split skill's input source from `/document/content` to
   `/document/merged_content`, reset and re-run the indexer, then prove
   `"depot maintenance window"` retrieves a chunk whose `title` is
   `service-notice.png`. (`indexer_client.reset_indexer(name)` clears the
   high-water mark so already-seen blobs are reprocessed.)

2. **Break the vectorizer deliberately.** Create a second index identical to this
   one but with `vector_search_dimensions=3072`, then run the same skillset against
   it. Capture the exact error and write down which three places the dimension must
   agree.

3. **Measure RRF.** For the query `"restocking fee"`, collect the top-10 chunk ids
   from keyword-only and from vector-only, compute the RRF score of each document
   yourself with $k=60$, and compare your ranking to what hybrid actually returned.
   Explain any document that appears in the hybrid results but in neither top-3.

4. **Scope the agent.** Add a `doc_type` filter so the agent can only see `.md`
   content, and confirm the SLA question now returns `NOT_IN_DOCUMENTS`. Which
   object did you change — the tool, the index, or the skillset?

In [ ]:
# Your work here

---

## Cleanup

Delete in dependency order: **indexer → skillsets → data source → index**. An index
with a live indexer pointed at it will not delete cleanly, and a data source in use
by an indexer is refused.

⚠️ **This does not stop your Azure AI Search bill.** The *service* is what bills
hourly, not the objects inside it. Deleting every index in a Basic service saves
you nothing. To actually stop the charge, delete the search service or the whole
resource group — see [99_teardown](../../99_teardown/README.md).

In [ ]:
def drop(label, fn, *args):
    try:
        fn(*args)
        print("deleted ", label)
    except Exception as exc:
        print("skipped ", label, "-", type(exc).__name__)


if agent_id:
    drop(f"thread {thread_id}", project.agents.threads.delete, thread_id)
    drop(f"agent {agent_id}", project.agents.delete_agent, agent_id)

drop(f"indexer {INDEXER_NAME}", indexer_client.delete_indexer, INDEXER_NAME)
drop(f"skillset {SKILLSET_NAME}", indexer_client.delete_skillset, SKILLSET_NAME)
drop(f"skillset {OCR_SKILLSET_NAME}", indexer_client.delete_skillset, OCR_SKILLSET_NAME)
drop(
    f"data source {DATASOURCE_NAME}",
    indexer_client.delete_data_source_connection,
    DATASOURCE_NAME,
)
drop(f"index {INDEX_NAME}", search_index_client().delete_index, INDEX_NAME)
drop(f"container {CONTAINER}", blob_service_client().delete_container, CONTAINER)

print("\nStill billing: the Azure AI Search SERVICE itself, hourly.")
print("Stop it with 99_teardown, or leave it only if you are moving straight to 05.2.")

## Check yourself

[quiz.md](quiz.md) — 14 questions on the pipeline object model, field attributes,
skills, projections, retrieval modes, and agent wiring.

## Next

[05.2 — Extract content from documents](../02_document_extraction/README.md)